<a href="https://colab.research.google.com/github/allengr220/cognitive-ledger/blob/main/longevity/baseline_risk_builder_v1_split_aware_export_levinenocrp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline Risk Builder V1 — Split-Aware Export for Levine-noCRP Benchmark Cohort

This notebook applies the frozen BaselineRisk builder to the **Levine-noCRP aligned benchmark cohort** and exports the baseline-scored train/test artifacts required for downstream PhysioRisk and benchmark comparison notebooks.

## Purpose

For the benchmark cohort restricted to the shared no-CRP biomarker set, this notebook:

1. loads the benchmark cohort dataset
2. creates the analysis train/test split
3. fits the frozen BaselineRisk model **on the training split only**
4. applies the fitted baseline model unchanged to both train and test
5. exports:
   - train parquet with `BaselineRisk`
   - test parquet with `BaselineRisk`
   - fitted baseline model artifact
   - metadata JSON describing split and model configuration

## Context

This notebook is part of the aligned benchmark pipeline comparing:

- reduced-panel PhysioRisk (benchmark-aligned variant)
- TotalRisk derived from that physiology score
- Levine-style no-CRP comparator models built on the same cohort

The cohort used here is not the full canonical PhysioRisk feature set. It is a restricted benchmark cohort designed to preserve cohort alignment for downstream comparison.

## Frozen baseline specification

- Formula: `cr(RIDAGEYR, df=4) + C(RIAGENDR)`
- Model: Cox proportional hazards
- Penalizer: `0.01`
- `l1_ratio = 0.0`
- Duration column: `PERMTH_INT`
- Event column: `MORTSTAT`

## Architectural invariants

- **BaselineRisk is fit on train only and then frozen.**
- **The fitted baseline model is applied unchanged to both train and test.**
- **No downstream notebook may refit baseline.**
- **This notebook changes the cohort, not the baseline architecture.**

## Interpretation

`BaselineRisk` represents the age/sex mortality component for the Levine-noCRP benchmark cohort. It is expected to be strongly age-correlated and to capture the dominant baseline hazard structure, while downstream physiology models are responsible for modeling residual mortality risk beyond baseline age/sex effects.

## Goal

Produce a clean, split-aware, frozen baseline export for the Levine-noCRP benchmark cohort so that all downstream models operate on a common baseline foundation without leakage or architectural drift.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install lifelines

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from patsy import dmatrix, build_design_matrices
from lifelines import CoxPHFitter


## Configuration

This notebook assumes a **single cohort parquet** with all rows, then applies a temporal split.

If you already have explicit train/test cohort files, adapt the load/split cell accordingly.


In [ ]:
# ============================================================
# USER CONFIG
# ============================================================

# Input cohort parquet (all rows for a given analysis cohort)
COHORT_PATH = "/content/drive/MyDrive/LRE/data/pub/benchmark_levine_no_crp_cohort.parquet"

# Output directory for this specific baseline export
OUT_DIR = Path("/content/drive/MyDrive/LRE/data/pub/baseline/levine_benchmark")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Output file names
TRAIN_OUT_PATH = OUT_DIR / "physiorisk_v1_frozen_baseline_train_levinenocrp.parquet"
TEST_OUT_PATH  = OUT_DIR / "physiorisk_v1_frozen_baseline_test_levinenocrp.parquet"
MODEL_PATH     = OUT_DIR / "baseline_risk_v1_temporal_fit_levinenocrp.pkl"
META_PATH      = OUT_DIR / "baseline_risk_v1_temporal_metadata_levinenocrp.json"

# Split settings
SPLIT_COL = "cycle_start_year"
TRAIN_MAX_YEAR = 2010
TEST_MIN_YEAR = 2011

# Frozen baseline spec
BASELINE_FORMULA = "cr(RIDAGEYR, df=4) + C(RIAGENDR)"
PENALIZER = 0.01
L1_RATIO = 0.0
DURATION_COL = "PERMTH_INT"
EVENT_COL = "MORTSTAT"

# Core required cols
ID_COL = "SEQN"
AGE_COL = "RIDAGEYR"
SEX_COL = "RIAGENDR"

print("COHORT_PATH:", COHORT_PATH)
print("OUT_DIR:", OUT_DIR)


## Load cohort


In [ ]:
df = pd.read_parquet(COHORT_PATH).copy()

required_cols = [ID_COL, AGE_COL, SEX_COL, DURATION_COL, EVENT_COL, SPLIT_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=required_cols).copy()
df[AGE_COL] = pd.to_numeric(df[AGE_COL], errors="coerce")
df = df.dropna(subset=[AGE_COL]).copy()

assert df[ID_COL].is_unique, f"{ID_COL} is not unique in cohort parquet"

print("Loaded cohort:", df.shape)
print("Deaths:", int(df[EVENT_COL].sum()))
print("Split years:", sorted(df[SPLIT_COL].dropna().unique().tolist())[:10], "...")


## Build temporal split


In [ ]:
train_df = df[df[SPLIT_COL] <= TRAIN_MAX_YEAR].copy()
test_df  = df[df[SPLIT_COL] >= TEST_MIN_YEAR].copy()

print("Train shape:", train_df.shape)
print("Train deaths:", int(train_df[EVENT_COL].sum()))
print("Test shape:", test_df.shape)
print("Test deaths:", int(test_df[EVENT_COL].sum()))

if len(train_df) == 0 or len(test_df) == 0:
    raise ValueError("Train or test split is empty. Check split settings.")


## Design-matrix helpers

These use the exact frozen baseline specification.


In [ ]:
def build_baseline_design(df, formula, design_info=None):
    if design_info is None:
        X = dmatrix(formula + " - 1", df, return_type="dataframe")
        return X, X.design_info
    else:
        X = build_design_matrices([design_info], df, return_type="dataframe")[0]
        X = pd.DataFrame(X, index=df.index, columns=design_info.column_names)
        return X, design_info


## Fit baseline on train only


In [ ]:
X_train, design_info = build_baseline_design(train_df, BASELINE_FORMULA, design_info=None)
design_columns = list(X_train.columns)

fit_train_df = pd.concat(
    [train_df[[DURATION_COL, EVENT_COL]], X_train],
    axis=1
).copy()

cph = CoxPHFitter(penalizer=PENALIZER, l1_ratio=L1_RATIO)
cph.fit(
    fit_train_df,
    duration_col=DURATION_COL,
    event_col=EVENT_COL
)

print("Fitted coefficient index:")
print(list(cph.summary.index))
cph.summary.loc[:, ["coef", "exp(coef)", "se(coef)", "p"]]


## Apply frozen baseline to train and test


In [ ]:
X_train_scoring, _ = build_baseline_design(train_df, BASELINE_FORMULA, design_info=design_info)
X_test_scoring, _  = build_baseline_design(test_df, BASELINE_FORMULA, design_info=design_info)

train_df = train_df.copy()
test_df = test_df.copy()

train_df["BaselineRisk"] = cph.predict_log_partial_hazard(X_train_scoring).values
test_df["BaselineRisk"]  = cph.predict_log_partial_hazard(X_test_scoring).values

train_df["BaselinePartialHazard"] = np.exp(train_df["BaselineRisk"])
test_df["BaselinePartialHazard"]  = np.exp(test_df["BaselineRisk"])

print("Train BaselineRisk vs age corr:")
print(train_df[[AGE_COL, "BaselineRisk"]].corr())

print("\nTest BaselineRisk vs age corr:")
print(test_df[[AGE_COL, "BaselineRisk"]].corr())


## Quick sanity checks


In [ ]:
assert "BaselineRisk" in train_df.columns
assert "BaselineRisk" in test_df.columns

print("Train BaselineRisk summary:")
print(train_df["BaselineRisk"].describe())

print("\nTest BaselineRisk summary:")
print(test_df["BaselineRisk"].describe())


## Save artifacts


In [ ]:
joblib.dump(cph, MODEL_PATH)

metadata = {
    "notebook_role": "split_aware_baseline_export",
    "cohort_path": COHORT_PATH,
    "split_col": SPLIT_COL,
    "train_max_year": TRAIN_MAX_YEAR,
    "test_min_year": TEST_MIN_YEAR,
    "formula": BASELINE_FORMULA,
    "duration_col": DURATION_COL,
    "event_col": EVENT_COL,
    "penalizer": PENALIZER,
    "l1_ratio": L1_RATIO,
    "design_columns": design_columns,
    "n_train": int(len(train_df)),
    "n_train_events": int(train_df[EVENT_COL].sum()),
    "n_test": int(len(test_df)),
    "n_test_events": int(test_df[EVENT_COL].sum()),
    "baseline_refit_downstream_allowed": False,
}

with open(META_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

train_df.to_parquet(TRAIN_OUT_PATH, index=False)
test_df.to_parquet(TEST_OUT_PATH, index=False)

print("Saved:")
print("  model:", MODEL_PATH)
print("  metadata:", META_PATH)
print("  train parquet:", TRAIN_OUT_PATH)
print("  test parquet:", TEST_OUT_PATH)


In [ ]:
from lifelines.utils import concordance_index

# Train
cindex_train = concordance_index(
    train_df[DURATION_COL],
    -train_df["BaselineRisk"],  # negative because higher risk = shorter survival
    train_df[EVENT_COL]
)

# Test
cindex_test = concordance_index(
    test_df[DURATION_COL],
    -test_df["BaselineRisk"],
    test_df[EVENT_COL]
)

print(f"BaselineRisk C-index (train): {cindex_train:.4f}")
print(f"BaselineRisk C-index (test):  {cindex_test:.4f}")

In [ ]:
from lifelines.utils import concordance_index

cindex_age_only = concordance_index(
    test_df["PERMTH_INT"],
    -test_df["RIDAGEYR"],
    test_df["MORTSTAT"]
)

print(f"Age-only C-index (test): {cindex_age_only:.4f}")

## Notes

- This notebook is the correct upstream producer for the canonical frozen-baseline PhysioRisk notebook.
- If you need a benchmark cohort or CRP cohort, duplicate this notebook and change only:
  - `COHORT_PATH`
  - `OUT_DIR`
  - split settings if needed

## Invariant

After these artifacts are created, all downstream notebooks should treat `BaselineRisk` as fixed.
